**Original code**: [Babak Khavari](https://github.com/babakkhavari)<br>
**Conceptualization & Methodological review** : [Babak Khavari](https://github.com/babakkhavari)<br>
**Updates, Modifications**: [UIEP SEForALL](uiep@seforall.org)<br>

# Clustering Notebook

The following notebook can be used in order to replicate the population clusters developed and published in [PopClusters](https://data.mendeley.com/datasets/z9zfhzk8cr/). Please see **Population cluster data to assess the urban-rural split and electrification in Sub-Saharan Africa** ([available here](https://www.nature.com/articles/s41597-021-00897-9)) for more information. For a more thorough description of each function please refer to the [functions.py](http://localhost:8888/notebooks/scripts) file. 

## Datasets
The cluster makes use of three (3) GIS-datasets:
* **Administrative units (vector polygon)** - This should be disagreggated. It will be used to 1) delimit the population layer to the area of interest and 2) to limit the maximum size of the clusters
* **Population (raster)**
* **Nighttime lights (raster)** - This will be used in order to estimate electrified population in each cluster.


## Pre-processing
Before using this notebook please ensure that all of your datasets are in the WGS84 coordinate reference system (EPSG:4326) and that your raster datasets are clipped to the administrative boundaries of the area of interest 


## Output
The final clusters will include 7 columns.

1. **id** – The IDs are given as a unique number for each cluster. This enables the user to process the data contained in the clusters outside of a GIS software and then merge the data with the clusters.


2. **Country** – Name of the country. 


3. **Population** – This is the population in each cluster obtained from the population dataset.


4. **NightLight** – This value is obtained from the nighttime light map and represents the maximum luminance detected in each cluster (best results are obtained with stable lights).


5. **ElecPop** – The number of people in each cluster who live in areas where light sources are detected.


6. **Area** – The area of each cluster given in square kilometres.


7. **IsUrban** - Urban/Peri-urban/Rural classification (urban = 2, peri-urban = 1, rural = 0)
    

## Cell 1 - Importing packages

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent
sys.path.insert(0, str(root_dir))
from scripts.functions import *

## Cell 2 - Selecting Datasets

Select the workspace, this is the folder that will be used for the outputs. 

**NOTE** Select an empty folder as all the files will be deleted from the workspace once the clusters are generated

You will also have to select the three datasets used in the analysis. These are: administrative boundaries, population (.tif), Nighttime lights (.tif)
 


In [ ]:
workspace = os.path.join(root_dir, 'data', 'outputs')

messagebox.showinfo('OnSSET', 'Select the population map')
filename_pop = filedialog.askopenfilename(initialdir=os.path.join(root_dir, 'data', 'inputs', 'pop'), 
                                          filetypes=(("rasters",["*.tif", "*.nc"]),("all files","*.*")))
poprasterio=rasterio.open(filename_pop)
nodata_pop = poprasterio.nodata
pop=gdal.Open(filename_pop)

In [ ]:
messagebox.showinfo('OnSSET', 'Select the admin map')
filename_admin = (filedialog.askopenfilename(initialdir=os.path.join(root_dir, 'data', 'inputs', 'admin_boundaries'),
                                             filetypes=(("vector", ["*.shp", "*.geojson", "*.gpkg", "*.parquet"]),("all files","*.*"))))
admin=gpd.read_file(filename_admin)

## Cell 3 - Setting study area name

This will dictate the name displayed in the country column of the final clusters

In [ ]:
study_area_name = "Mozambique"

## Cell 4 - Setting the target coordinate system
When calculating distances and areas it is important to choose a coordinate system that represents distances and areas correctly in your area of interst.

In order to select your own coordinate system go to [epsg.io](http://epsg.io/) and type in your area of interest, this will give you a list of coordinate systems to choose from. Once you have selected your coordinate system replace the numbers below with the numbers from your coordinate system **(keep the "EPSG" part)**.

**NOTE** When selecting your coordinate system make sure that you select a system with the unit of meters (or another linear lenght unit), this is indicated for all systems on [epsg.io](http://epsg.io/)

In [ ]:
crs = 'EPSG:32736'

## Cell 5 - Filter thresholds

Enter thresholds for population and nighttime light. All raster cells with values under the threshold in the population map will be removed.

In [ ]:
population_threshold = 0

## Cell 6 - Clipping raster layers

Clipping the population map to the extent of the study area.

In [ ]:
pop_clipped_path = clipRasterByExtent(os.path.join(workspace, study_area_name + "Pop.tif"), pop, admin, nodata_pop)

## Cell 7 - Reclassifying rasters

Reclassifies the clipped population layer. The function sets everything under the thresholds to zero. The first parameter is the clipped raster from cell 6 and the second parameter the thresholds from cell 5.

In [ ]:
reclassified_Pop = reclassifyRasters(pop_clipped_path, population_threshold)

## Cell 8 - Resample population raster

Resample population layer and save the resampled map to disc.

The **resample_factor** is the factor used in the resampling (i.e if you have a raster with cell size 30m a factor 3 creates an output raster with cell size 90m). You are recommended to keep this value as 1 if the cell-size off your population layer is larger than or equal to 100 x 100 meter.

In [ ]:
resample_factor = 1

In [ ]:
resampled_Pop = resampleRaster(reclassified_Pop, 1)
saveRaster(resampled_Pop, os.path.join(workspace, "rasterBase.tif"))

## Cell 9 - Creating the cluster base

This step creates the clusters

In [ ]:
rasterize(admin, filename_admin, resampled_Pop, os.path.join(workspace, 'raster_admin.tif'))
rasterMultiplication(os.path.join(workspace, "rasterBase.tif"), os.path.join(workspace, "raster_admin.tif"), os.path.join(workspace, "rasterBase.tif"))
clusters = toPolygon(os.path.join(workspace, "rasterBase.tif"), os.path.join(workspace, "clusters"))

## Cell 10 (Optional) - Beautifying the cluster base
**beautifying_clusters** refines the geometry of polygon raster shapes to enhance visual representation. The user can modify two inputs: 
* *distance* is a distance threshold for smoothing
* *alpha* is an alpha value for shape simplification

distance = 25 and alpha = 0.01 have been found to create good clusters shapes for Mozambique when using the 100x100m WorldPop raster inputs. 

**Note that this process can take many hours depending on the computer!**

In [ ]:
distance = 25
alpha = 0.01

In [ ]:
#clusters = beautyfing_clusters(clusters, distance, alpha, crs)

## Cell 11 - Adding attributes to clusters

Generates the *id*, *Country* and *Area* columns.

In [ ]:
clusters = addAttributes(clusters, crs, study_area_name)

## Cell 12 - Populating clusters with data

Adding the *Population* column.

In [ ]:
clusters = zonal_stats_exact(pop_clipped_path, clusters, name='Population', method='sum')
clusters.Population.fillna(0, inplace=True)

## Cell 13 - Calculate Degree of Urbanization (DEGURBA)

In [ ]:
input_data = resample_raster_sum(pop_clipped_path, os.path.join(workspace, "ResampledPop2.tif"), scale_factor=5)
clusters = zonal_stats_exact(os.path.join(workspace, "ResampledPop2.tif"), clusters, name='Density', method='mean')
clusters = calculate_degurba(clusters, os.path.join(workspace, "ResampledPop2.tif"))

## Cell 14 - Remove very small clusters

Any settlement with a population lower than *threshold* will be removed and excluded from the analysis

In [ ]:
threshold = 3  

In [ ]:
clusters2 = clusters.loc[clusters.Population > 3]
clusters2 = clusters2[['id', 'Country', 'Area', 'Population', 'DEGURBA', 'geometry']]

## Cell 15 - Save results as clusters.parquet

In [ ]:
clusters2.to_parquet(os.path.join(workspace, "clusters.parquet"), index=False)
finished()
print('')
for i in [os.path.join(workspace, study_area_name + "Pop.tif"), os.path.join(workspace, 'raster_admin.tif'), os.path.join(workspace, "rasterBase.tif"), 
          os.path.join(workspace, "ResampledPop2.tif"), os.path.join(workspace, "clusters.dbf"), os.path.join(workspace, "clusters.prj"),
         os.path.join(workspace, "clusters.shx"), os.path.join(workspace, "clusters.shp")]:
    try:
        os.remove(i)
    except Exception as e:
        print(e)